In [ ]:
# Copyright (C) 2026 Advanced Micro Devices, Inc. All rights reserved.
#
# SPDX-License-Identifier: BSD-3-Clause
#
# Portions of this file consist of AI-generated content. AI-assisted
# content has been reviewed and validated by the authors.

# The brain on the iGPU: Llama 3.2 3B via llama.cpp + ROCm

The robot's brain is Llama 3.2 3B Instruct (Q4_K_M), running on the
Ryzen APU's iGPU through a `llama-server` built with ROCm/HIP
(`-DGGML_HIP=ON`). In the pipeline its one job is turning transcripts
into machine-readable intents - but that contract (the GBNF grammar, the
routing rules) is the intent node's business and lives in
`02_ros2/03_intent_node.ipynb`. THIS notebook is about the iGPU tenant
itself: bringing the server up with the right flags, verifying it, and
talking to it.

The AMD-specific skill here is **TODO 1.5**: the launch flags and
environment that keep a Strix APU alive.

The LLM runs in its own process, and everything talks to it over
localhost HTTP. That gives the iGPU one tenant with a clean lifecycle,
keeps the model warm across commands, and lets you develop against a
server someone else started.


In [ ]:
# workshop bootstrap: run against the finished reference code in solution/
# (no TODO stubs), so every cell works out of the box. To exercise YOUR
# implementations instead, change "solution" to "project" and restart.
import sys, pathlib
import os
import time

PROJECT = (pathlib.Path.cwd() / ".." / ".." / "solution").resolve()
assert (
    PROJECT / "config" / "workshop.yaml"
).exists(), f"solution not found at {PROJECT}"
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))

from common.quiet import hush_benign_warnings

hush_benign_warnings()  # silence known-benign third-party log noise

from common.config import load_config, resolve, PROJECT_ROOT
import requests
import ipywidgets as widgets
from IPython.display import display
from jupyter_ui_poll import ui_events
import subprocess, time, os

cfg = load_config()
print("running against:", PROJECT_ROOT)

## 1. The launch recipe

**The server is already started for you.** `./run_notebooks.sh` launches it
alongside Jupyter when the binary and the model are present (look for
"Starting llama-server on the iGPU" in its output; the log is
`workshop/logs/llama-server.log`, and the server shuts down when you quit
Jupyter).

If you ever need to start it yourself, run this from the **repository
root** in a terminal and leave it running:

```bash
HSA_OVERRIDE_GFX_VERSION=11.5.0 \
third_party/llama.cpp/build/bin/llama-server \
  --model models/llama-3.2-3b/Llama-3.2-3B-Instruct-Q4_K_M.gguf \
  --host 127.0.0.1 --port 8081 \
  --n-gpu-layers 99 --ctx-size 2048 --parallel 1 --no-warmup
```

First load takes 10-30 s while 3B quantized parameters stream into iGPU
memory. What each piece is for:

| piece | why it's there |
|---|---|
| `--n-gpu-layers 99` | put *every* layer on the iGPU; that's the point |
| `--ctx-size 2048` | **must stay capped.** The model's native 131072-token context would allocate a KV cache that OOMs the iGPU's VRAM carve-out (`cudaMalloc failed: out of memory`, then a segfault). Intent parsing needs ~100 tokens. |
| `--parallel 1` | one slot = one KV cache |
| `--no-warmup` | the warmup pass can sometimes segfault |
| `HSA_OVERRIDE_GFX_VERSION` | the ROCm runtime may not recognize the iGPU natively; presenting it as its nearest supported relative makes kernel dispatch deterministic. `11.5.0` Strix Point, `11.5.1` Strix Halo, `11.0.0` the broad fallback. Use `env.setdefault`, never clobber a value the user already exported. |

The same command, composed from the config (this is TODO 1.5's shape):

In [ ]:
l = cfg["llm"]
cmd = [
    str(resolve(l["server_bin"])),
    "--model",
    str(resolve(l["model_gguf"])),
    "--host",
    l["host"],
    "--port",
    str(l["port"]),
    "--n-gpu-layers",
    str(l["n_gpu_layers"]),
    "--ctx-size",
    str(l["ctx_size"]),
    "--parallel",
    str(l.get("parallel", 1)),
] + (["--no-warmup"] if l.get("no_warmup") else [])

env = os.environ.copy()
if l.get("hsa_override_gfx_version"):
    env.setdefault("HSA_OVERRIDE_GFX_VERSION", str(l["hsa_override_gfx_version"]))

print("HSA_OVERRIDE_GFX_VERSION =", env.get("HSA_OVERRIDE_GFX_VERSION"))
print("\nThe fully resolved command:\n")
print(
    "  HSA_OVERRIDE_GFX_VERSION=%s \\\n  %s"
    % (env.get("HSA_OVERRIDE_GFX_VERSION"), " \\\n    ".join(cmd))
)

> **Process-reality check.** The server is a background process; the
> notebook only ever plays *client*. If it misbehaves, watch
> `workshop/logs/llama-server.log`.

## 2. Check the server is up

The health check is one HTTP GET. The cells below need a live server, so if
this reports the server is missing, start it (section 1) and re-run this
cell.

In [ ]:
BASE = f"http://{l['host']}:{l['port']}"

WORKSHOP_DIR = (pathlib.Path.cwd() / ".." / "..").resolve()
LOG_PATH = str(WORKSHOP_DIR / "llama_server.log")
START_SCRIPT = str(WORKSHOP_DIR / "start_llama_server.sh")


def server_alive(base):
    try:
        return requests.get(f"{base}/health", timeout=0.5).status_code == 200
    except requests.RequestException:
        return False


def start_server(base, wait=10):
    SERVER_UP = server_alive(base)
    if SERVER_UP:
        print(f"llama-server already up at {base}: Llama 3.2 3B on the iGPU")
        return

    print(f"no llama-server at {base}. Starting in background...")
    log = open(LOG_PATH, "w")
    proc = subprocess.Popen(
        ["sh", START_SCRIPT],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )
    print(f"launched pid {proc.pid}, tailing log for ~{wait}s...\n")

    # follow the log file for `wait` seconds, then move on
    deadline = time.time() + wait
    with open(LOG_PATH, "r") as f:
        while time.time() < deadline:
            line = f.readline()
            if line:
                print(line, end="")
            else:
                if server_alive(base):
                    print(f"\nserver ready at {base}")
                    return
                time.sleep(0.2)

    ready = "ready" if server_alive(base) else "still booting"
    print(f"\nmoving on ({ready}). full log at {LOG_PATH}")


start_server(BASE)

## 3. Talk to it

The server speaks the standard chat-completions API, so "hello world" is
one HTTP POST. First a canned prompt, then a console: whatever you type
goes to the model on the iGPU and the reply is printed with its latency.
Press **done** to release the cell and move on.


In [ ]:
if not server_alive(BASE):
    raise RuntimeError(
        "no llama-server running. Start it (section 1), "
        "re-run the health check in section 2, then this cell."
    )

http = requests.Session()  # keep-alive: saves ~5-20 ms per turn


def chat(prompt, max_tokens=192, temperature=0.7):
    """One plain chat completion on the iGPU - no grammar, no routing."""
    payload = {
        "messages": [
            {
                "role": "system",
                "content": "You are a concise assistant running on the iGPU "
                "of an AMD Ryzen AI machine.",
            },
            {"role": "user", "content": prompt},
        ],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    r = http.post(f"{BASE}/v1/chat/completions", json=payload, timeout=60)
    r.raise_for_status()
    return r.json()["choices"][0]["message"]["content"].strip()


print(chat("In one sentence: what are you and what are you running on?"))

In [ ]:
if not server_alive(BASE):
    raise RuntimeError(
        "no llama-server running. Start it (section 1), "
        "re-run the health check in section 2, then this cell."
    )

box = widgets.Text(
    placeholder="ask anything, e.g. why do robots like grammars?",
    continuous_update=False,  # sync the value once, on Enter/blur
    layout=widgets.Layout(width="480px"),
)
send_btn = widgets.Button(description="send", button_style="primary")
done_btn = widgets.Button(description="done", button_style="danger")
console = widgets.Output(layout=widgets.Layout(border="1px solid #444", padding="6px"))
finished = {"flag": False}
done_btn.on_click(lambda _: finished.update(flag=True))


def send(_=None):
    text = box.value.strip()
    if not text:
        return
    box.value = ""
    with console:
        t0 = time.perf_counter()
        reply = chat(text)
        ms = 1e3 * (time.perf_counter() - t0)
        print(f"you  : {text}")
        print(f"llama: {reply}   ({ms:.0f} ms)\n")


send_btn.on_click(send)
# Enter (or clicking away) commits the text; the observer sends it.
box.observe(send, "value")
display(widgets.VBox([widgets.HBox([box, send_btn, done_btn]), console]))

with ui_events() as poll:
    while not finished["flag"]:
        poll(10)  # keep send/done clicks flowing
        time.sleep(0.05)
print("console closed")

Expect a few hundred milliseconds for a short answer: ~100 tokens through
a 3B Q4_K_M model on the iGPU. The pipeline itself never chats like this;
it sends one-line transcripts and reads back one-line JSON intents,
grammar-constrained so the output *cannot* be malformed. That whole
contract - the GBNF grammar, temperature 0, and the stop fast path that
bypasses the LLM entirely - belongs to the intent node:
`02_ros2/03_intent_node.ipynb`.

## 4. Your turn

Open **`project/models/llm.py`** and implement:

| TODO | What | Taught in |
|---|---|---|
| **1.5** | `llama_server_command(cfg)` -> `(cmd, env)` | section 1 |
| **1.6** | `parse_intent(...)` -> `ParsedCommand` | `02_ros2/03_intent_node.ipynb` (the grammar + HTTP contract) |

Then, from the `project/` directory:

```bash
python -m models.selftest

# to see all the TODOs you have left
python app.py
```

The harness validates your launch recipe offline (fake binaries, real
assertions, including the ctx-size cap and the HSA override) and runs your
`parse_intent` against a local test server, *inspecting the payload you
sent*: no grammar attached = named feedback, not a mystery.

That's the iGPU tenant up and talking. Next: `02_ros2/01_cameras.ipynb`,
where things start talking to each other.
